In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(len(os.listdir('/content/ambitious/ml_fun/images_houses')))

249


In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

drive_dataset_path = "/content/drive/MyDrive/ambitious/ml_fun"
local_dataset_path = "/content/new/"

# Ensure destination directory exists
os.makedirs(local_dataset_path, exist_ok=True)

# Get all file paths
file_list = []
for root, _, files in os.walk(drive_dataset_path):
    for file in files:
        file_list.append(os.path.join(root, file))

# Define a function to copy a single file
def copy_file(src):
    rel_path = os.path.relpath(src, drive_dataset_path)
    dest_path = os.path.join(local_dataset_path, rel_path)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    shutil.copy2(src, dest_path)

# Copy files in parallel
with ThreadPoolExecutor() as executor:
    list(tqdm(executor.map(copy_file, file_list), total=len(file_list), desc="Copying files", unit="file"))


Copying files: 100%|██████████| 24030/24030 [04:31<00:00, 88.66file/s] 


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

class PreprocessedBuildingDataset(Dataset):
    def __init__(self, dataframe, preprocessed_dir, transform=None, min_year=1900, max_year=2024, bin_width=30):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing metadata (e.g., 'bouwjaar').
            preprocessed_dir (str): Path to folder with preprocessed images.
            transform (callable, optional): Transform to be applied to each image.
            min_year (int): The minimum year for binning.
            max_year (int): The maximum year for binning.
            bin_width (int): The width of each bin in years.
        """
        self.df = dataframe.reset_index(drop=True)
        self.preprocessed_dir = preprocessed_dir
        self.transform = transform  # Store the passed transform
        self.min_year = int(min_year)
        self.max_year = int(max_year)
        self.bin_width = int(bin_width)
        self.num_bins = ((self.max_year - self.min_year) // self.bin_width) + 1

        # Compute labels
        self.df['label'] = ((self.df['bouwjaar'] - self.min_year) // self.bin_width).clip(0, self.num_bins - 1)
        self.labels = self.df['label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """
        Returns:
            image (torch.Tensor): Transformed image.
            label (torch.Tensor): Class label.
        """
        img_path = self.df.iloc[idx]['image_path']
        path = os.path.join('/content/new', img_path)
        label = self.df.iloc[idx]['label']

        # Try loading the image
        try:
            if not os.path.exists(path):
                raise FileNotFoundError(f"Image not found at {path}")

            image = Image.open(path).convert("RGB")
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            # Return a default tensor if image loading fails
            image = torch.zeros((3, 224, 224))

        return image, torch.tensor(label, dtype=torch.long)


In [ ]:
# Load Data
import pandas as pd

from torchvision import models
import torch.nn as nn
import torch.optim as optim
from torchvision.models import ResNet18_Weights
from torch.utils.data import Dataset, DataLoader, random_split

import numpy as np

from sklearn.utils.class_weight import compute_class_weight

import torch
from torchvision import models
from torchvision import transforms

# Load the data
df_loaded = pd.read_pickle('/content/new/database.pkl')

# Define the directory containing preprocessed images
preprocessed_dir = "/content/ambitious/images - Copy"


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Create the dataset and apply transforms
full_dataset = PreprocessedBuildingDataset(df_loaded, preprocessed_dir, min_year=1900, max_year=2024, bin_width=10)

# Split the dataset into training, validation, and test sets
train_size = int(0.8 * len(full_dataset))
val_size = (len(full_dataset) - train_size) // 2
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

import numpy as np

class_counts = np.bincount(full_dataset.labels)  # Count the number of samples per class
print(f"Class Counts: {class_counts}")

from torch.utils.data import DataLoader, WeightedRandomSampler

# Extract labels from the train dataset
train_labels = [full_dataset.labels[i] for i in train_dataset.indices]

# Recompute class weights and sample weights for the training dataset
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels]

# Create a WeightedRandomSampler for the training dataset
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),  # Length of the train dataset
    replacement=True
)

# Apply different transforms to the datasets
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_test_transform
test_dataset.dataset.transform = val_test_transform


print(f"✅ Dataset Split: {train_size} train, {val_size} val, {test_size} test.")


Class Counts: [1301 1308 2035 1839  713 2195 1922 2806 4178 2065 1794 1361  512]
✅ Dataset Split: 19223 train, 2403 val, 2403 test.


In [ ]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

def create_model(num_classes):
    """
    Create an EfficientNet-B0 model with dropout and a customized final layer.

    Args:
        num_classes (int): Number of output classes.

    Returns:
        torch.nn.Module: Customized EfficientNet-B0 model.
    """
    # Load EfficientNet-B0 with pretrained weights
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last two feature blocks (for more learning capacity)
    for param in model.features[-8:].parameters():
        param.requires_grad = True

    # Customize the final fully connected layer with dropout
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(model.classifier[1].in_features, num_classes)
    )

    return model

# Create the model
model = create_model(num_classes=full_dataset.num_bins)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Compute class weights for imbalanced classes
class_weights = compute_class_weight(
    class_weight='balanced', classes=np.arange(full_dataset.num_bins), y=full_dataset.labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Define loss function with class weights and label smoothing
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

import torch.optim as optim

# Define different learning rates and weight decay for specific layers
optimizer = optim.Adam([
    {'params': model.features[:4].parameters(), 'lr': 2.5e-6, 'weight_decay': 1e-6},  # Lower LR and L2 for early layers
    {'params': model.features[4:].parameters(), 'lr': 5e-6, 'weight_decay': 1e-5},  # Higher LR and L2 for mid-level layers
    {'params': model.classifier.parameters(), 'lr': 1e-4, 'weight_decay': 1e-4}     # Highest LR for classifier layers
])



from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)



/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, scheduler=None, scheduler_step_on='epoch', epochs=15):
    import torch.optim.lr_scheduler

    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    for epoch in range(epochs):
        # Log current learning rate
        current_lr = optimizer.param_groups[0]['lr']
        print(f'\nEpoch {epoch+1}/{epochs} | Current Learning Rate: {current_lr:.6f}')

        # Training Phase
        model.train()
        running_loss = 0.0
        train_progress = tqdm(train_loader, desc="Training", leave=False)

        for inputs, labels in train_progress:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            train_progress.set_postfix(loss=loss.item())

        train_loss = running_loss / len(train_loader)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        val_progress = tqdm(val_loader, desc="Validating", leave=False)

        with torch.no_grad():
            for inputs, labels in val_progress:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

                val_progress.set_postfix(loss=loss.item())

        val_loss = val_loss / len(val_loader)
        val_acc = correct / total

        # Update training history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

        # Save the best model based on validation accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/content/new/best_model.pth")

        # *** Scheduler Step ***
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                # Pass validation loss to ReduceLROnPlateau
                scheduler.step(val_loss)
            else:
                scheduler.step()  # For other schedulers like StepLR or CosineAnnealingLR

        # Save training history
        with open('/content/new/training_history.pkl', 'wb') as f:
            pickle.dump(history, f)

    return history


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pickle
import os

# Define batch size
batch_size = 32

# DataLoaders for training and validation
# Define the training DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


# # Check if there is a previously saved model
# model_path = "/content/test/best_model.pth"
# if os.path.exists(model_path):
#     print(f"Loading saved model from {model_path}...")
#     model.load_state_dict(torch.load(model_path, map_location=device))

print("🚀 Start training...")

# Train the model
history = train_model(model, train_loader, val_loader, optimizer, criterion,scheduler=scheduler, scheduler_step_on='epoch', epochs=30)

print(f"✅ Training complete")


🚀 Start training...

Epoch 1/30 | Current Learning Rate: 0.000003


Train Loss: 2.6073 | Val Loss: 2.5109 | Val Acc: 0.1939

Epoch 2/30 | Current Learning Rate: 0.000003


Train Loss: 2.4709 | Val Loss: 2.3830 | Val Acc: 0.2451

Epoch 3/30 | Current Learning Rate: 0.000003


Train Loss: 2.3607 | Val Loss: 2.2869 | Val Acc: 0.2855

Epoch 4/30 | Current Learning Rate: 0.000003


Train Loss: 2.2763 | Val Loss: 2.2160 | Val Acc: 0.3050

Epoch 5/30 | Current Learning Rate: 0.000003


...

Train Loss: 1.7612 | Val Loss: 2.0006 | Val Acc: 0.3891

Epoch 27/30 | Current Learning Rate: 0.000003


Train Loss: 1.7466 | Val Loss: 1.9935 | Val Acc: 0.3870

Epoch 28/30 | Current Learning Rate: 0.000003


Train Loss: 1.7381 | Val Loss: 1.9974 | Val Acc: 0.3862

Epoch 29/30 | Current Learning Rate: 0.000003


Train Loss: 1.7101 | Val Loss: 1.9946 | Val Acc: 0.3820

Epoch 30/30 | Current Learning Rate: 0.000001


Train Loss: 1.7054 | Val Loss: 1.9921 | Val Acc: 0.3866
✅ Training complete


In [ ]:
full_dataset.num_bins

13

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Define the test DataLoader
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Load the saved model
model_path = "/content/new/best_model.pth"
print(f"Loading model from {model_path}...")
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)

# Set the model to evaluation mode
model.eval()

# Variables to store predictions and ground truth
all_preds = []
all_labels = []

# Disable gradient calculation for evaluation
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        # Store predictions and labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate classification metrics
print("🔍 Evaluating performance on test dataset...")
print("\n📊 Classification Report:")
print(classification_report(all_labels, all_preds))

print("\n🟦 Confusion Matrix:")
conf_matrix = confusion_matrix(all_labels, all_preds)
print(conf_matrix)


<ipython-input-8-1e37456373b0>:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


Loading model from /content/new/best_model.pth...
🔍 Evaluating performance on test dataset...

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.25      0.48      0.33       130
           1       0.28      0.33      0.30       126
           2       0.31      0.22      0.26       193
           3       0.37      0.32      0.34       172
           4       0.22      0.38      0.27        77
           5       0.52      0.40      0.45       208
           6       0.41      0.37      0.39       198
           7       0.38      0.41      0.39       276
           8       0.56      0.42      0.48       419
           9       0.37      0.37      0.37       214
          10       0.40      0.30      0.35       189
          11       0.46      0.48      0.47       149
          12       0.19      0.44      0.26        52

    accuracy                           0.38      2403
   macro avg       0.36      0.38      0.36      2403
weighted avg  

/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.11.0 and strictly below 2.14.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.18.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're u

ModuleNotFoundError: No module named 'keras.src.engine'

In [ ]:
!pip install onnx onnx_tf